# Netflix Content Strategy & Business Insights

## Task 6: End-to-End Data Science Project

### Project Overview
In this project, I integrate data analysis, machine learning, interactive visualization, and business storytelling to explore Netflix's content catalogue and generate evidence-based business insights.

I build on my existing data preparation, exploratory data analysis, trend analysis, and content-type classification work to develop a unified business insights solution.

### Business Problem
Understanding the composition and distribution of a streaming platform's content catalogue can help stakeholders explore content diversity, geographic representation, genre patterns, and changes in catalogue additions over time.

This project uses Netflix catalogue data to investigate these patterns and demonstrate how descriptive analytics and machine learning can support structured business decision-making.

### Project Objectives
1. Analyse Netflix's content catalogue and identify meaningful patterns.
2. Evaluate machine learning models that classify titles as Movies or TV Shows.
3. Develop interactive dashboards using Streamlit, Plotly, and Power BI.
4. Design a PostgreSQL database and a Flask API to support data access and model predictions.
5. Translate analytical findings into evidence-based business recommendations.
6. Present the methodology, findings, limitations, and recommendations in a professional report.

### Tools and Technologies
- **Python:** Data analysis and machine learning
- **pandas and NumPy:** Data preparation and analysis
- **scikit-learn:** Model development and evaluation
- **Plotly and Streamlit:** Interactive Python dashboard
- **Power BI:** Business intelligence reporting
- **PostgreSQL:** Relational data storage and SQL analysis
- **Flask:** REST API development
- **Jupyter Notebook:** Analysis and documentation
- **Git and GitHub:** Version control and project presentation

### Data Source
The analysis uses a cleaned Netflix catalogue dataset containing 8,790 titles. The dataset includes information such as content type, release year, country, rating, genre, and date added to Netflix.

### Project Scope and Limitations
The dataset supports analysis of catalogue composition and recorded content characteristics. It does not, by itself, establish audience demand, viewing performance, revenue, or profitability.

The classification model predicts whether a title is a Movie or TV Show. Its predictions are not interpreted as forecasts of content popularity or commercial success.

In [15]:

# Import libraries for data analysis and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display more columns when inspecting the dataset
pd.set_option("display.max_columns", None)

# Use a consistent plotting style
sns.set_theme(style="whitegrid")

In [16]:
# Load the cleaned dataset from the project directory
df = pd.read_csv("netflix_cleaned.csv")

# Confirm the dataset size and available fields
print(f"Number of titles: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

display(df.head())

Number of titles: 8,790
Number of columns: 16


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,date_added_year,date_added_month,date_added_day,duration_value,duration_unit,genre_count
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries,2021,9,25,90,min,1
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",2021,9,24,1,Season,3
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",2021,9,24,1,Season,3
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",2021,9,22,91,min,2
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",2021,9,24,125,min,3


In [17]:
# Summarize missing values in each column
missing_summary = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2)
}).sort_values("Missing Values", ascending=False)

print("\nMISSING VALUE SUMMARY")
display(missing_summary)

# Count fully duplicated rows
print(f"\nFully duplicated rows: {df.duplicated().sum():,}")



MISSING VALUE SUMMARY


,Missing Values,Missing Percentage
director,2588,29.44
country,287,3.27
type,0,0.00
show_id,0,0.00
title,0,0.00
date_added,0,0.00
release_year,0,0.00
rating,0,0.00
duration,0,0.00
listed_in,0,0.00



Fully duplicated rows: 0


In [18]:
print(df.dtypes)

show_id               str
type                  str
title                 str
director              str
country               str
date_added            str
release_year        int64
rating                str
duration              str
listed_in             str
date_added_year     int64
date_added_month    int64
date_added_day      int64
duration_value      int64
duration_unit         str
genre_count         int64
dtype: object


In [19]:
df['date_added'].head(10)

0    2021-09-25
1    2021-09-24
2    2021-09-24
3    2021-09-22
4    2021-09-24
5    2021-09-24
6    2021-09-24
7    2021-05-01
8    2021-09-23
9    2021-05-01
Name: date_added, dtype: str

In [20]:
# Convert date_added from strings to pandas datetime
df["date_added"] = pd.to_datetime(
    df["date_added"],
    errors="coerce"
)

In [21]:
df['date_added'].dtypes

dtype('<M8[us]')

In [22]:
df.dtypes

show_id                        str
type                           str
title                          str
director                       str
country                        str
date_added          datetime64[us]
release_year                 int64
rating                         str
duration                       str
listed_in                      str
date_added_year              int64
date_added_month             int64
date_added_day               int64
duration_value               int64
duration_unit                  str
genre_count                  int64
dtype: object

In [23]:

# Calculate headline catalogue metrics
total_titles = len(df)
content_type_counts = df["type"].value_counts()
unique_countries = df["country"].dropna().nunique()

kpi_summary = pd.DataFrame({
    "Metric": [
        "Total titles",
        "Movies",
        "TV Shows",
        "Distinct country entries",
        "Earliest release year",
        "Latest release year"
    ],
    "Value": [
        total_titles,
        content_type_counts.get("Movie", 0),
        content_type_counts.get("TV Show", 0),
        unique_countries,
        df["release_year"].min(),
        df["release_year"].max()
    ]
})

display(kpi_summary)


,Metric,Value
0,Total titles,8790
1,Movies,6126
2,TV Shows,2664
3,Distinct country entries,85
4,Earliest release year,1925
5,Latest release year,2021


### Catalog Composition

I will examine the balance between Movies and TV Shows in the Netflix catalog. This comparison establishes the catalog's overall composition and provides a starting point for exploring differences in content types.

The distribution describes the titles included in this dataset; it does not indicate audience preferences or viewing demand.

In [24]:
import sys
!{sys.executable} -m pip install plotly


In [25]:
import sys
!{sys.executable} -m pip install "nbformat>=4.2.0"


In [26]:
import nbformat
import plotly.express as px

print("nbformat:", nbformat.__version__)
print("Plotly:", px.__version__ if hasattr(px, "__version__") else "Imported successfully")


nbformat: 5.11.1
Plotly: Imported successfully


In [27]:
import plotly
import plotly.express as px

# Calculate title counts and percentages by content type
type_summary = (
    df["type"]
    .value_counts()
    .rename_axis("Content Type")
    .reset_index(name="Title Count")
)

type_summary["Percentage"] = (
    type_summary["Title Count"] / len(df) * 100
).round(2)

display(type_summary)

,Content Type,Title Count,Percentage
0,Movie,6126,69.69
1,TV Show,2664,30.31


In [46]:

import plotly.express as px
import plotly.io as pio

# Display interactive charts in the browser.
pio.renderers.default = "browser"

# Summarize the catalogue by content type.
type_summary = (
    df["type"]
    .value_counts()
    .rename_axis("Content Type")
    .reset_index(name="Title Count")
)

# Calculate each content type's share of the catalogue.
type_summary["Percentage"] = (
    type_summary["Title Count"] / len(df) * 100
).round(2)

# Use consistent colors across the project.
content_colors = {
    "Movie": "#3B82F6",
    "TV Show": "#14B8A6"
}

# Build the interactive bar chart.
fig = px.bar(
    type_summary,
    x="Content Type",
    y="Title Count",
    color="Content Type",
    color_discrete_map=content_colors,
    text="Title Count",
    custom_data=["Percentage"],
    title="Netflix Catalogue by Content Type",
    labels={
        "Content Type": "Content Type",
        "Title Count": "Number of Titles"
    }
)

# Configure labels and hover details.
fig.update_traces(
    texttemplate="%{y:,}",
    textposition="outside",
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Titles: %{y:,}<br>"
        "Catalogue share: %{customdata[0]:.2f}%"
        "<extra></extra>"
    )
)

# Polish the chart layout.
fig.update_layout(
    template="plotly_white",
    xaxis_title="Content Type",
    yaxis_title="Number of Titles",
    showlegend=False,
    hovermode="closest",
    uniformtext_minsize=10,
    uniformtext_mode="hide"
)

# Display the interactive visualization.
fig.show()


### Catalog Addition Trends

In this section, I will also analyze the number of titles added to Netflix over time and compare trends between Movies and TV Shows. This helps identify years with relatively high or low catalog additions and determine whether the two content types follow similar patterns.

The analysis uses the `date_added_year` column, which records when a title was added to the dataset. These counts describe the available catalogue data and should not be interpreted as complete historical acquisition records or evidence of Netflix's future content strategy.

In [47]:

# Keep records with a known addition year and content type.
additions_df = df.dropna(
    subset=["date_added_year", "type"]
).copy()

# Ensure the addition year is stored as an integer.
additions_df["date_added_year"] = (
    additions_df["date_added_year"].astype(int)
)

# Count titles added in each year, separated by content type.
yearly_additions = (
    additions_df
    .groupby(["date_added_year", "type"])
    .size()
    .reset_index(name="Titles Added")
    .sort_values("date_added_year")
)

# Create the interactive yearly trend chart.
fig = px.line(
    yearly_additions,
    x="date_added_year",
    y="Titles Added",
    color="type",
    color_discrete_map=content_colors,
    markers=True,
    custom_data=["type"],
    title="Netflix Catalogue Additions by Year and Content Type",
    labels={
        "date_added_year": "Year Added",
        "Titles Added": "Number of Titles",
        "type": "Content Type"
    }
)

# Show detailed information when hovering over a data point.
fig.update_traces(
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Year added: %{x}<br>"
        "Titles added: %{y:,}"
        "<extra></extra>"
    )
)

# Format the chart for readability.
fig.update_layout(
    template="plotly_white",
    xaxis_title="Year Added",
    yaxis_title="Number of Titles",
    hovermode="closest",
    legend_title_text="Content Type"
)

# Display the interactive chart in your browser.
fig.show()


### Key Findings: Catalogue Addition Trends

- **Movie additions peaked in 2019**, with 1,424 titles added, representing the highest annual count of Movie additions in the available dataset.
- **TV Show additions peaked in 2020**, with 595 titles added, representing the highest annual count of TV Show additions in the available dataset.
- The peaks occurred in different years, suggesting that the timing of the highest recorded additions varied between the two content types.

**Business implication:** These patterns provide a historical view of catalogue additions that can support further investigation into content-mix changes over time. However, the data alone does not explain why these peaks occurred or establish their effect on viewership, engagement, or revenue.

### Analyzing Netflix Content Characteristics

In this section, I examine the characteristics of titles in the Netflix catalogue, focusing on content ratings, duration, and genre distribution. These indicators help describe the types of content represented in the dataset and identify differences between Movies and TV Shows.

The analysis describes catalogue metadata rather than audience suitability, viewing behaviour, or content performance.

In [48]:

# Count titles by content rating and type.
rating_summary = (
    df.dropna(subset=["rating", "type"])
    .groupby(["rating", "type"])
    .size()
    .reset_index(name="Title Count")
)

# Order ratings by their total number of titles.
rating_order = (
    rating_summary.groupby("rating")["Title Count"]
    .sum()
    .sort_values(ascending=False)
    .index.tolist()
)

# Compare rating distributions across content types.
fig = px.bar(
    rating_summary,
    x="rating",
    y="Title Count",
    color="type",
    barmode="group",
    category_orders={"rating": rating_order},
    color_discrete_map=content_colors,
    title="Netflix Titles by Content Rating and Type",
    labels={
        "rating": "Content Rating",
        "Title Count": "Number of Titles",
        "type": "Content Type"
    },
    hover_data={"Title Count": ":,"}
)

# Format the interactive chart.
fig.update_layout(
    template="plotly_white",
    xaxis_title="Content Rating",
    yaxis_title="Number of Titles",
    xaxis_tickangle=-45,
    legend_title_text="Content Type",
    hovermode="closest"
)

fig.show()


### Key Findings: Content Ratings

- **Movies:** TV-MA is the most frequent content rating, with 2,062 titles.
- **TV Shows:** TV-MA is also the most frequent rating, with 1,143 titles.
- **Shared pattern:** TV-MA is the leading rating for both content types, indicating that this rating represents a substantial part of the available catalogue.

**Business implication:** Content ratings can help describe the composition of the catalogue and support content segmentation. However, these counts alone do not establish audience preferences, viewing demand, or the suitability of titles for particular viewers.

### Duration Analysis

I examine the recorded duration of Movies and TV Shows to understand the structure of the Netflix catalogue. Movies are generally measured in minutes, while TV Shows are recorded in seasons. Because these units are different, I analyse them separately rather than comparing them on the same numerical scale.

In [50]:

# Count TV Shows by number of seasons.
season_counts = (
    tv_shows_df["duration_value"]
    .value_counts()
    .sort_index()
    .rename_axis("Number of Seasons")
    .reset_index(name="Title Count")
)

display(season_counts)

# Show the most common season count.
print("Most common season count:")
display(season_counts.loc[[season_counts["Title Count"].idxmax()]])


,Number of Seasons,Title Count
0,1,1791
1,2,421
2,3,198
3,4,94
4,5,64
5,6,33
6,7,23
7,8,17
8,9,9
9,10,6


Most common season count:


,Number of Seasons,Title Count
0,1,1791


In [49]:

# Separate Movies and TV Shows because their duration units differ.
movies_df = df[
    (df["type"] == "Movie") &
    (df["duration_unit"] == "min") &
    (df["duration_value"].notna())
].copy()

tv_shows_df = df[
    (df["type"] == "TV Show") &
    (df["duration_unit"] == "Season") &
    (df["duration_value"].notna())
].copy()

# Compare the distribution of Movie runtimes.
fig_movies = px.histogram(
    movies_df,
    x="duration_value",
    nbins=30,
    color_discrete_sequence=["#3B82F6"],
    title="Distribution of Movie Runtimes",
    labels={
        "duration_value": "Runtime (minutes)",
        "count": "Number of Movies"
    },
    hover_data=["duration_value"]
)

fig_movies.update_layout(
    template="plotly_white",
    xaxis_title="Runtime (minutes)",
    yaxis_title="Number of Movies"
)

fig_movies.show()

# Compare the distribution of TV Show season counts.
fig_tv = px.histogram(
    tv_shows_df,
    x="duration_value",
    nbins=20,
    color_discrete_sequence=["#14B8A6"],
    title="Distribution of TV Show Seasons",
    labels={
        "duration_value": "Number of Seasons",
        "count": "Number of TV Shows"
    }
)

fig_tv.update_layout(
    template="plotly_white",
    xaxis_title="Number of Seasons",
    yaxis_title="Number of TV Shows"
)

fig_tv.show()


### Key Findings: TV Show Season Distribution

- **Single-season shows dominate:** 1,791 TV Shows are recorded as having one season, making this the most frequent season count in the dataset.
- **Counts decline with additional seasons:** The number of titles generally decreases as the recorded season count increases.
- **Business implication:** This distribution describes how TV Shows are represented in the catalogue. It does not establish whether shows were cancelled, completed, or successful.

### Examining the genre analysis
This section will examine the genres associated with Netflix titles to identify the most common content categories and compare genre representation between Movies and TV Shows.
Because the listed_in column can contain multiple genres in a single row, we’ll split those entries so each genre can be counted individually. This will help us understand catalog composition and identify content categories that may warrant further business analysis.

In [53]:

# Split multi-genre entries so each genre can be counted separately.
genre_data = df[["type", "listed_in"]].dropna().copy()

genre_data["listed_in"] = genre_data["listed_in"].str.split(", ")
genre_data = genre_data.explode("listed_in")

# Count titles within each genre and content type.
genre_summary = (
    genre_data.groupby(["listed_in", "type"])
    .size()
    .reset_index(name="Title Count")
)

# Display the 15 most frequently represented genres overall.
top_genres = (
    genre_summary.groupby("listed_in")["Title Count"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .index
)

top_genre_summary = genre_summary[
    genre_summary["listed_in"].isin(top_genres)
]

display(
    top_genre_summary.sort_values(
        "Title Count", ascending=False
    )
)


,listed_in,type,Title Count
16,International Movies,Movie,2752
12,Dramas,Movie,2426
7,Comedies,Movie,1674
17,International TV Shows,TV Show,1349
10,Documentaries,Movie,869
0,Action & Adventure,Movie,859
34,TV Dramas,TV Show,762
15,Independent Movies,Movie,756
4,Children & Family Movies,Movie,641
24,Romantic Movies,Movie,616


In [54]:

# Compare the most common genres across Movies and TV Shows.
fig = px.bar(
    top_genre_summary,
    x="Title Count",
    y="listed_in",
    color="type",
    barmode="group",
    orientation="h",
    color_discrete_map=content_colors,
    title="Top 15 Netflix Genres by Content Type",
    labels={
        "listed_in": "Genre",
        "Title Count": "Number of Titles",
        "type": "Content Type"
    },
    hover_data={"Title Count": ":,"}
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Number of Titles",
    yaxis_title="Genre",
    yaxis={"categoryorder": "total ascending"},
    legend_title_text="Content Type",
    hovermode="closest"
)

fig.show()


### Key Findings: Genre Distribution

- **International Movies** is the most represented genre in the catalogue, with 2,752 titles, followed by **Dramas** (2,426) and **Comedies** (1,674).
- Among TV Shows, **International TV Shows** leads with 1,349 titles, followed by **TV Dramas** (762) and **TV Comedies** (573).
- Movies have strong representation across several categories, including Documentaries (869), Action & Adventure (859), and Independent Movies (756).
- TV Shows also have notable representation in Crime TV Shows (469), Kids' TV (448), and Docuseries (394).

**Business implication:** The catalogue contains a broad mix of international, dramatic, comedic, documentary, and other content categories. These patterns can help guide further analysis of catalogue composition and content segmentation. However, genre counts alone do not establish audience demand, viewing performance, profitability, or unmet content needs.

**Data interpretation note:** A single title can be assigned to multiple genres. Therefore, genre counts overlap and should not be summed to estimate the number of unique titles.

### Geographic Distribution of Netflix Content

This section examines the countries associated with Netflix titles to understand the geographic composition of the catalogue. Identifying the most frequently represented countries can help describe the catalogue's international reach and highlight differences between Movies and TV Shows.

Because some titles are associated with multiple countries, the `country` column must be split before counting. Missing or placeholder country entries should be excluded from the country rankings so they do not distort the results.

**Business question:** Which countries are most frequently represented in Netflix's catalogue, and how does representation differ between Movies and TV Shows?

In [55]:

# Keep country and content type, excluding missing and placeholder entries.
country_data = df[["country", "type"]].copy()

country_data["country"] = country_data["country"].replace(
    ["Not Given", ""], np.nan
)

country_data = country_data.dropna(subset=["country"])

# Split multi-country entries into individual country records.
country_data["country"] = country_data["country"].str.split(", ")
country_data = country_data.explode("country")

# Remove accidental whitespace around country names.
country_data["country"] = country_data["country"].str.strip()

# Count titles associated with each country and content type.
country_summary = (
    country_data.groupby(["country", "type"])
    .size()
    .reset_index(name="Title Count")
)

# Identify the 15 countries with the highest total representation.
top_countries = (
    country_summary.groupby("country")["Title Count"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .index
)

top_country_summary = country_summary[
    country_summary["country"].isin(top_countries)
]

display(
    top_country_summary.sort_values(
        "Title Count", ascending=False
    )
)


,country,type,Title Count
128,United States,Movie,2395
47,India,Movie,976
129,United States,TV Show,845
126,United Kingdom,Movie,387
85,Pakistan,TV Show,350
127,United Kingdom,TV Show,251
15,Canada,Movie,187
60,Japan,TV Show,172
108,South Korea,TV Show,165
34,France,Movie,148


In [56]:

# Compare Movie and TV Show representation across the top 15 countries.
fig = px.bar(
    top_country_summary,
    x="Title Count",
    y="country",
    color="type",
    barmode="group",
    orientation="h",
    color_discrete_map=content_colors,
    title="Top 15 Countries Represented in Netflix's Catalogue",
    labels={
        "country": "Country",
        "Title Count": "Number of Titles",
        "type": "Content Type"
    },
    hover_data={"Title Count": ":,"}
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Number of Country–Title Associations",
    yaxis_title="Country",
    yaxis={"categoryorder": "total ascending"},
    legend_title_text="Content Type",
    hovermode="closest"
)

fig.show()


### Key Findings: Geographic Distribution of Netflix Content Types

- **The United States leads the catalogue** with 2,395 Movie associations and 845 TV Show associations, making it the most represented country for both content types in this analysis.
- **India ranks second for Movies**, with 976 associations, but has 81 TV Show associations in the displayed results.
- **The United Kingdom** contributes 387 Movie and 251 TV Show associations, showing representation across both content types.
- **Pakistan** has 350 TV Show associations compared with 71 Movie associations, while **Japan** has 172 TV Show associations compared with 87 Movie associations.
- Other countries represented in the top-country results include South Korea, France, Spain, Egypt, Nigeria, Mexico, Turkey, Germany, Canada, and Australia.

**Business implications:** The results show that Netflix's catalog includes content associated with multiple countries, with particularly strong representation from the United States and India. Differences between Movie and TV Show counts may help inform further investigation of regional catalog composition and content strategy.

**Limitations:** These counts represent country–title associations, not exclusive production origins. A title associated with multiple countries contributes to each country's count. The data describes catalog representation, not audience demand, viewing hours, profitability, or the performance of regional content.



### Release-Year Analysis

This section examines the original release years of Movies and TV Shows in the Netflix catalogue. Unlike the content-addition analysis, which shows when titles were added to Netflix, release-year analysis helps us understand the age and recency of the content represented in the dataset.

Comparing release-year distributions across Movies and TV Shows can reveal whether the catalogue is concentrated in recent releases, older titles, or a mixture of both.

**Business question:** How does the release-year distribution differ between Movies and TV Shows, and how strongly is the catalogue represented by recent releases?

In [57]:

# Check the range and distribution of original release years.
print("Earliest release year:", df["release_year"].min())
print("Latest release year:", df["release_year"].max())

release_summary = (
    df.groupby(["release_year", "type"])
    .size()
    .reset_index(name="Title Count")
)

display(release_summary.head())


Earliest release year: 1925
Latest release year: 2021


,release_year,type,Title Count
0,1925,TV Show,1
1,1942,Movie,2
2,1943,Movie,3
3,1944,Movie,3
4,1945,Movie,3


In [58]:

# Count releases by year and content type.
fig = px.line(
    release_summary,
    x="release_year",
    y="Title Count",
    color="type",
    color_discrete_map=content_colors,
    markers=True,
    title="Netflix Catalogue by Original Release Year",
    labels={
        "release_year": "Original Release Year",
        "Title Count": "Number of Titles",
        "type": "Content Type"
    },
    hover_data={"Title Count": ":,"}
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Original Release Year",
    yaxis_title="Number of Titles",
    legend_title_text="Content Type",
    hovermode="closest"
)

fig.show()


In [59]:

# Group each title into a decade based on its original release year.
decade_df = df[["release_year", "type"]].dropna().copy()
decade_df["Release Decade"] = (
    (decade_df["release_year"] // 10) * 10
).astype(int).astype(str) + "s"

decade_summary = (
    decade_df.groupby(["Release Decade", "type"])
    .size()
    .reset_index(name="Title Count")
)

decade_order = sorted(
    decade_summary["Release Decade"].unique(),
    key=lambda decade: int(decade[:-1])
)

fig_decades = px.bar(
    decade_summary,
    x="Release Decade",
    y="Title Count",
    color="type",
    barmode="group",
    category_orders={"Release Decade": decade_order},
    color_discrete_map=content_colors,
    title="Netflix Catalogue by Original Release Decade",
    labels={
        "Release Decade": "Release Decade",
        "Title Count": "Number of Titles",
        "type": "Content Type"
    },
    hover_data={"Title Count": ":,"}
)

fig_decades.update_layout(
    template="plotly_white",
    xaxis_title="Release Decade",
    yaxis_title="Number of Titles",
    legend_title_text="Content Type",
    hovermode="closest"
)

fig_decades.show()


In [60]:
# checking which release years has the most titles
# Rank release years by the number of titles in the catalogue.
top_release_years = (
    release_summary
    .groupby("release_year")["Title Count"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .rename_axis("Release Year")
    .reset_index()
)

display(top_release_years)


,Release Year,Title Count
0,2018,1146
1,2017,1030
2,2019,1030
3,2020,953
4,2016,901
5,2021,592
6,2015,555
7,2014,352
8,2013,286
9,2012,236


In [61]:
# checking which decades dominates

# Summarize the number of titles associated with each release decade.
decade_totals = (
    decade_summary
    .groupby("Release Decade")["Title Count"]
    .sum()
    .sort_values(ascending=False)
    .rename_axis("Release Decade")
    .reset_index()
)

display(decade_totals)


,Release Decade,Title Count
0,2010s,5913
1,2020s,1545
2,2000s,807
3,1990s,274
4,1980s,129
5,1970s,70
6,1960s,25
7,1940s,15
8,1950s,11
9,1920s,1


### Key Findings: Original Release-Year Distribution

- **The catalogue is concentrated in recent releases.** The 2010s account for 5,913 titles, making this the most represented decade, followed by the 2020s with 1,545 titles and the 2000s with 807 titles.
- **2018 is the most represented individual release year**, with 1,146 titles. The years 2017 and 2019 each account for 1,030 titles, followed by 2020 with 953 titles.
- Representation decreases across older decades, from 274 titles in the 1990s to 129 in the 1980s and 70 in the 1970s. The dataset also contains a small number of titles released before 1970.

**Business implications:** The results indicate that the Netflix catalogue represented in this dataset is weighted toward titles released during the 2010s and early 2020s. This provides a useful overview of the catalogue's content age profile and can support further investigation into how release recency relates to genre, content type, or other available characteristics.

**Limitations:** These results describe titles present in the dataset, not all content originally released during each period. Release-year counts do not establish viewing demand, audience preferences, or commercial success. Release year should also be distinguished from the year a title was added to Netflix.

## Content Characteristics by Type

This section summarises the structural characteristics of Movies and TV Shows in the Netflix catalogue. Movie runtime is measured in minutes, while TV Show duration is recorded in seasons. Analysing these separately helps describe the typical format of each content type.

**Business question:** What do movie runtimes and TV Show season counts reveal about the structure of Netflix's catalogue?

In [62]:

# Summarise movie runtimes in minutes.
movie_duration_summary = movies_df["duration_value"].describe().round(2)

print("MOVIE RUNTIME SUMMARY (MINUTES)")
display(movie_duration_summary.to_frame(name="Value"))

# Summarise TV Show duration in seasons.
tv_duration_summary = tv_shows_df["duration_value"].describe().round(2)

print("TV SHOW SEASON SUMMARY")
display(tv_duration_summary.to_frame(name="Value"))

# Identify the most common recorded season count.
print("MOST COMMON TV SHOW SEASON COUNT")
display(
    tv_shows_df["duration_value"]
    .value_counts()
    .head(1)
    .rename_axis("Number of Seasons")
    .to_frame(name="Title Count")
)


MOVIE RUNTIME SUMMARY (MINUTES)


,Value
count,6126.00
mean,99.58
std,28.28
min,3.00
25%,87.00
50%,98.00
75%,114.00
max,312.00


TV SHOW SEASON SUMMARY


,Value
count,2664.00
mean,1.75
std,1.55
min,1.00
25%,1.00
50%,1.00
75%,2.00
max,17.00


MOST COMMON TV SHOW SEASON COUNT


,Title Count
Number of Seasons,
1,1791


### Key Findings: Movie Runtime and TV Show Duration

- **Movie runtime:** The 6,126 Movies have an average runtime of approximately 99.58 minutes and a median of 98 minutes. The middle 50% of movie runtimes fall between 87 and 114 minutes, indicating that many movies in the catalogue are around 1.5–2 hours long.
- **Variation in movie length:** Recorded runtimes range from 3 to 312 minutes. These extremes may represent unusually short or long titles and could be investigated further to confirm whether they are valid entries.
- **TV Show duration:** Across 2,664 TV Shows, the average recorded length is 1.75 seasons, while the median is 1 season. The 25th percentile is also 1 season, and the 75th percentile is 2 seasons.
- **Single-season shows dominate:** A total of 1,791 TV Shows are recorded as having one season, making this the most common duration category. The longest recorded TV Show has 17 seasons.

**Business implications:** The catalogue contains Movies with a typical runtime of roughly 100 minutes, while TV Shows are more commonly represented by shorter season counts. These differences can inform content-format segmentation and further analysis of catalogue composition.

**Limitations:** Duration describes the structure of titles, not their popularity, viewing completion, profitability, or quality. The minimum and maximum movie runtimes should be reviewed before drawing conclusions about unusually short or long content. Season counts also do not reveal the number of episodes or total viewing time.

### Data quality checks for duration
Checking the extreme movie runtimes of 3 minutes and 312 minutes to see which titles they correspond to.

In [63]:

# Inspect the shortest and longest recorded movie runtimes.
movie_duration_extremes = movies_df[
    movies_df["duration_value"].isin([
        movies_df["duration_value"].min(),
        movies_df["duration_value"].max()
    ])
][["title", "duration_value", "duration"]].sort_values(
    "duration_value"
)

display(movie_duration_extremes)


,title,duration_value,duration
2820,Silent,3,3 min
7822,Black Mirror: Bandersnatch,312,312 min


### Duration Data Quality Check

The runtime analysis identified two extreme values: *Silent*, recorded at 3 minutes, and *Black Mirror: Bandersnatch*, recorded at 312 minutes.

These observations warrant further review because unusually short runtimes and interactive titles may not be directly comparable with conventional movie runtimes. The records should be checked against the source data before any correction or exclusion is made.

For the current analysis, the recorded values have been retained to preserve the original dataset, as I acknowledge that extreme observations may affect summary statistics, particularly the mean and range.

## Business Recommendations

The analysis of Netflix's catalogue reveals patterns in content type, release year, genre, country representation, ratings, and duration. These findings provide a foundation for identifying opportunities for further investigation and improving content portfolio decisions.

The recommendations below are based on catalogue composition rather than direct viewing, engagement, or revenue data. They should therefore be treated as strategic directions to investigate, not as confirmed evidence of commercial performance.

### 1. Review the balance between Movies and TV Shows

Movies account for 6,126 titles, compared with 2,664 TV Shows. Netflix could evaluate whether this catalogue mix aligns with its content strategy and audience needs by combining catalogue data with viewing hours, completion rates, and subscriber engagement.

### 2. Evaluate genre representation alongside audience demand

International Movies, Dramas, and Comedies are among the most represented genres in the catalogue. Netflix could compare these categories with viewing performance, audience retention, and subscriber feedback to determine whether the current genre mix aligns with demand and where potential content gaps may exist.

### 3. Investigate geographic representation

The United States has the highest country-associated title counts in both Movies and TV Shows, while countries such as India, the United Kingdom, Pakistan, and Japan show different representation patterns across content types. Netflix could combine these catalogue patterns with regional viewing data and market research to assess whether its content portfolio reflects the needs of audiences in different markets.

### 4. Assess the role of release recency

Titles released during the 2010s form the largest release-decade group in this dataset, with 2018 being the most represented individual year. Netflix could examine viewing performance by release year to understand whether newer releases, older catalogue titles, or particular release periods contribute most to engagement and retention.

### 5. Review content metadata and duration outliers

The duration analysis identified movie runtimes ranging from 3 to 312 minutes, while most TV Shows have one or two recorded seasons. Netflix could validate unusual runtime records and improve metadata consistency to support more reliable reporting and content comparisons.

### 6. Incorporate viewing and engagement data into future analysis

Catalogue counts show what content is represented, but not what audiences watch or value. Integrating viewing hours, completion rates, repeat viewing, subscriber retention, and content acquisition or production costs would enable stronger assessments of content performance and investment priorities.

### Overall Recommendation

Netflix should use catalogue analysis as an initial portfolio assessment and combine it with audience, engagement, and cost data before making significant content investment decisions. This would help distinguish simple catalogue representation from actual audience demand and commercial value.

## Conclusion

This project explored Netflix's catalogue through descriptive analysis and interactive visualisations. The analysis examined content-type distribution, catalogue addition trends, content ratings, movie runtimes, TV Show season counts, genre representation, geographic distribution, and original release years.

The findings show that Movies outnumber TV Shows in the dataset, international and drama-related categories are strongly represented, the United States has the highest country-associated title counts, and the catalogue is concentrated in releases from the 2010s and early 2020s. The duration analysis also highlighted the importance of checking unusual values and interpreting Movies and TV Shows separately.

Although these patterns provide useful insights into catalogue composition, the dataset does not directly measure audience preferences, viewing engagement, or profitability. Future analysis incorporating viewing and subscriber data could support more reliable conclusions about content performance and strategic investment.

Overall, this project demonstrates how data cleaning, exploratory analysis, interactive visualisation, and evidence-based recommendations can support business understanding and decision-making.